In [1]:
import os 
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from typing import List

load_dotenv()

chat = ChatOllama(model="llama3.2", temperature=0.7)

In [2]:
curriculo = """
Nome: Kaiky Cruz de Sousa
Email: kaiky.cruz@email.com
Telefone: (11) 91234-5678

Educação:
- Graduação em Análise e Desenvolvimento de Sistemas, Universidade SPTech, 2023-2024
- Curso Formação Engenheiro de IA generativa, Udemy, 2026

Experiência Profissional:
- Desenvolvedor de Software Sênior, Google, 2025-2026
    - Responsável por líderar a equipe de desenvolvimento.
- Engenheiro de IA Sênior, Itaú, 2026
    - Responsável pelo desenvolvimento de agentes de IA
"""

1. Definir o esquema de saída com Pydantic

In [3]:
class Experiencia(BaseModel):
    """Um item de experiência profissional"""
    cargo: str = Field(..., description="O cargo ocupado.")
    empresa: str = Field(..., description="O nome da empresa.")
    periodo: str = Field(..., description="O período trabalhado na empresa.")

class Educacao(BaseModel):
    """um item de educação"""
    nivel: str = Field(..., description="Nível da formação da pessoa.")
    universidade: str = Field(..., description="Universidade que a pessoa estudou.")
    periodo: str = Field(..., description="Período que a pessoa estudou.")

class Curriculo(BaseModel):
    """Informações extraídas de um currículo"""
    nome: str = Field(..., description="Nome completo da pessoa.")
    email: str = Field(..., description="Email da pessoa.")
    telefone: str = Field(..., description="Número de telefone da pessoa.")
    educacao: List[Educacao] = Field(..., description="Lista de instituições que a pessoa estudou.")
    experiencia: List[Experiencia] = Field(..., description="Lista da experiência profissional da pessoa.")


parser_curriculo = JsonOutputParser(pydantic_object=Curriculo)

2. Apresentando o Prompt com instruções do parser

In [4]:
prompt_textual = f"""
Sua tarefa é extrair informações chave de um currículo e formatá-las como um objeto JSON.
As instruções de formatação JSON estão abaixo. Certifique-se de seguir o formato e os tipos de dados exatamente como especificado.
{parser_curriculo.get_format_instructions()}

Texto do Currículo:
{curriculo}
"""

3. Invocar o LLM e analisar a resposta

In [5]:
resposta_curriculo = chat.invoke(prompt_textual)
resultado_curriculo = parser_curriculo.parse(resposta_curriculo.content)

4. Usar o Parser para analisar a resposta

In [6]:
print("--- Análise do Currículo ---")
# Use a notação de colchetes para acessar os valores do dicionário
print(f"Nome: {resultado_curriculo['nome']}")
print(f"Email: {resultado_curriculo['email']}")
print(f"Telefone: {resultado_curriculo['telefone']}")

print("\nEducação:")
for edu in resultado_curriculo['educacao']:
    print(f"- {edu['nivel']} na {edu['universidade']} ({edu['periodo']})")

print("\nExperiência Profissional:")
for exp in resultado_curriculo['experiencia']:
    print(f"- {exp['cargo']} na {exp['empresa']} ({exp['periodo']})")

--- Análise do Currículo ---
Nome: Kaiky Cruz de Sousa
Email: kaiky.cruz@email.com
Telefone: (11) 91234-5678

Educação:
- Graduação em Análise e Desenvolvimento de Sistemas na Universidade SPTech (2023-2024)
- Curso Formação Engenheiro de IA generativa na Udemy (2026)

Experiência Profissional:
- Desenvolvedor de Software Sênior na Google (2025-2026)
- Engenheiro de IA Sênior na Itaú (2026)
